In [1]:
from datasets import load_dataset
from src.config import cfg
import re
import string
import unicodedata
from collections import Counter

In [4]:
ds = load_dataset(cfg.dataset.path, cfg.dataset.name)

In [6]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)

    text = text.lower().replace("i\u0307", "i")

    text = re.sub(r"[\'\"`’‘“”]", "", text)

    # sayı-sayı arasındaki tire → boşluk
    text = re.sub(r"(?<=\d)-(?=\d)", " ", text)

    # harf-harf olmayan tireler → boşluk
    text = re.sub(r"(?<!\w)[-–—]|[-–—](?!\w)", " ", text)

    text = re.sub(r"[^\w\s\-çğıöşü]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
def process_data(ds=ds):
    train = ds["train"]#.select(range(100))

    def process_row(row):
        text = row["text"]
        text = clean_text(text)
        row["tokens"] = text.split()
        return row
        
    train = train.map(process_row, num_proc=8, load_from_cache_file=False)
    return train["text"], train["tokens"]
data,tokens=process_data()

In [ ]:
import os
import glob
import pandas as pd

OUTPUT = "data/trwiki.parquet"

files = sorted(
    f for f in glob.glob(
        "/tmp/data/extracted/**/*",
        recursive=True
    )
    if os.path.isfile(f)
)

all_chunks = []

for i, file in enumerate(files, start=1):
    print(f"[{i}/{len(files)}] loading {file}")
    try:
        for chunk in pd.read_json(file, lines=True, chunksize=10000):
            all_chunks.append(chunk)
    except Exception as e:
        print(f"[skip] {file}: {e}")

print("[concat] merging...")

df = pd.concat(all_chunks, ignore_index=True)

print(f"[save] rows={len(df)}")

df.to_parquet(OUTPUT, index=False)

print(f"[done] saved → {OUTPUT}")

In [43]:
import pandas as pd

df = pd.read_parquet("data/trwiki.parquet")

print("\nshape:")
print(df.shape)

print("\ncolumns:")
print(df.columns.tolist())

print("\nhead:")
print(df.head())

print("\ninfo:")
print(df.info())

print("\nsample text:")
print(df.iloc[0]["title"])
print(df.iloc[0]["text"][:1000])


shape:
(1005338, 5)

columns:
['id', 'revid', 'url', 'title', 'text']

head:
   id    revid                                     url                title  \
0  10  1703128  https://tr.wikipedia.org/wiki?curid=10           Cengiz Han   
1  16   221544  https://tr.wikipedia.org/wiki?curid=16  Film (anlam ayrımı)   
2  22  1160848  https://tr.wikipedia.org/wiki?curid=22        Mustafa Suphi   
3  24  1637189  https://tr.wikipedia.org/wiki?curid=24                Linux   
4  25   124718  https://tr.wikipedia.org/wiki?curid=25                  MHP   

                                                text  
0  Cengiz Han, (doğum adıyla Temuçin ya da Timuçi...  
1                       Film şu anlamlara gelebilir:  
2  Mehmed Mustafa Subhi (), kısaca Mustafa Suphi ...  
3  Linux (telaffuz: "Lin-uks"); Linux çekirdeğine...  
4                                                     

info:
<class 'pandas.DataFrame'>
RangeIndex: 1005338 entries, 0 to 1005337
Data columns (total 5 columns):
 #   Colu

In [45]:
import pandas as pd

df = pd.read_parquet("data/trwiki.parquet")

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

print(f"rows: {len(df):,}")
print(f"columns: {len(df.columns)}")

print("\ncolumns:")
print(df.columns)

print("\nmissing values:")
print(df.isnull().sum())

print("\nrandom samples:")
print(
    df.sample(5)[
        ["id", "title"]
    ]
)

print("\ntext length stats:")
text_len = df["text"].str.len()

print(
    text_len.describe()
)

print("\nfirst article:")
row = df.iloc[0]

print(
    f"title: {row['title']}"
)

print(
    f"text:\n{row['text'][:2000]}"
)

DATASET OVERVIEW
rows: 1,005,338
columns: 5

columns:
Index(['id', 'revid', 'url', 'title', 'text'], dtype='str')

missing values:
id       0
revid    0
url      0
title    0
text     0
dtype: int64

random samples:
             id                                   title
607967  2768919                 İskenderiyeli Sosigenes
784585  3481780  2022 Peru kendi kendine darbe girişimi
684547  3139480                            Pityokteines
263994  1259063                       Estadio belvedere
48164    161342                           Yuriy Bilonoh

text length stats:
count    1.005338e+06
mean     7.625639e+02
std      2.701963e+03
min      0.000000e+00
25%      0.000000e+00
50%      8.400000e+01
75%      5.390000e+02
max      1.979380e+05
Name: text, dtype: float64

first article:
title: Cengiz Han
text:
Cengiz Han, (doğum adıyla Temuçin ya da Timuçin, 1162 - Ağustos 1227), Moğol İmparatorluğu'nun kurucusu ve ilk Han'ı olan Moğol komutan ve hükümdardır. Hükümdarlığı döneminde gerçekleşt